# DermaTriage — Exploratory Data Analysis

Explore the harmonised training manifest: class balance, Fitzpatrick skin-type
coverage (with a focus on dark types IV–VI), sample images, and summary
statistics including the synthetic-vs-real split.

Run from `ml_pipeline/notebooks/` after `make data`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image

# Make the project package importable.
sys.path.insert(0, os.path.abspath('..'))

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

class_names = cfg['data']['class_names']
dark_types = cfg['data']['fitzpatrick_dark_types']
processed = cfg['data']['processed_path']

manifest_path = os.path.join('..', processed, 'train', 'manifest.csv')
df = pd.read_csv(manifest_path)
print(f'Loaded {len(df)} rows from {manifest_path}')
df.head()

## Class distribution

In [ ]:
counts = df['label_idx'].value_counts().sort_index()
labels = [class_names[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(labels, counts.values, color='#00695C')
ax.set_title('Training images per class')
ax.set_ylabel('Count')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Fitzpatrick skin-type distribution

All types, then dark types (IV–VI) highlighted separately. `-1` denotes images
with unknown Fitzpatrick metadata (to be joined in during data prep).

In [ ]:
fitz_counts = df['fitzpatrick_type'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# All Fitzpatrick types.
axes[0].bar([str(t) for t in fitz_counts.index], fitz_counts.values,
            color='#FF8F00')
axes[0].set_title('All Fitzpatrick types')
axes[0].set_xlabel('Fitzpatrick type')
axes[0].set_ylabel('Count')

# Dark types IV-VI only.
dark = fitz_counts[fitz_counts.index.isin(dark_types)]
axes[1].bar([str(t) for t in dark.index], dark.values, color='#4B2E1E')
axes[1].set_title(f'Dark Fitzpatrick types {dark_types}')
axes[1].set_xlabel('Fitzpatrick type')

plt.tight_layout()
plt.show()

dark_total = int(dark.sum())
known = int(fitz_counts[fitz_counts.index >= 1].sum())
if known:
    print(f'Dark-skin (IV-VI) share of known-type images: {dark_total/known:.1%}')

## Sample images per class

In [ ]:
n = len(class_names)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.2 * rows))
axes = axes.flatten()

for idx in range(n):
    ax = axes[idx]
    subset = df[df['label_idx'] == idx]
    ax.set_title(class_names[idx], fontsize=10)
    ax.axis('off')
    if len(subset):
        path = subset.iloc[0]['image_path']
        try:
            ax.imshow(Image.open(path).convert('RGB'))
        except (FileNotFoundError, OSError):
            ax.text(0.5, 0.5, 'image missing', ha='center', va='center')
    else:
        ax.text(0.5, 0.5, 'no samples', ha='center', va='center')

for j in range(n, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## Dataset statistics

In [ ]:
total = len(df)
n_synth = int(df['is_synthetic'].sum()) if 'is_synthetic' in df else 0
n_real = total - n_synth

print(f'Total images       : {total}')
print(f'Real images        : {n_real}')
print(f'Synthetic images   : {n_synth}')
if total:
    print(f'Synthetic : real    : {n_synth/total:.1%} : {n_real/total:.1%}')
if 'source' in df:
    print('\nImages by source:')
    print(df['source'].value_counts().to_string())

print('\nPer-class counts:')
for idx, count in df['label_idx'].value_counts().sort_index().items():
    print(f'  {class_names[idx]:<28} {count}')